## Setup and Imports
Importi:
transformers per tokenizer/modello/Trainer
torch per training/inferenza
utility (json, os, numpy, pandas, tqdm)

In [1]:
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import pandas as pd
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


In [2]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

# Model configuration
model_name = "dmis-lab/biobert-v1.1"  # BioBERT for biomedical text
output_model_dir = "../models/bert_biomedbert_ner_label_weight"

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']

Model: dmis-lab/biobert-v1.1
Output directory: models/bert_biomedbert_ner_label_weight


## Data Loading Functions

In [3]:
def load_ner_data(file_paths):
    """
    Load NER data from multiple JSON files.
    Each file contains documents with entities.
    """
    all_data = {}
    
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)}")
        else:
            print(f"Warning: {file_path} not found")
    
    return all_data


def prepare_documents_for_ner(data):
    """
    Convert raw data into structured format for NER.
    Each document has title and abstract as separate text segments.
    """
    documents = []
    
    for pmid, article in data.items():
        # Process title
        title_text = article['metadata']['title']
        title_entities = [e for e in article['entities'] if e['location'] == 'title']
        
        documents.append({
            'pmid': pmid,
            'location': 'title',
            'text': title_text,
            'entities': title_entities
        })
        
        # Process abstract
        abstract_text = article['metadata']['abstract']
        abstract_entities = [e for e in article['entities'] if e['location'] == 'abstract']
        
        documents.append({
            'pmid': pmid,
            'location': 'abstract',
            'text': abstract_text,
            'entities': abstract_entities
        })
    
    return documents


print("✓ Data loading functions defined")

✓ Data loading functions defined


## Load Training and Dev Data

In [4]:
import json
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[1]
DATA_ROOT = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2025"
ANNOTATIONS_DIR = DATA_ROOT / "Annotations"

train_files = [
    ANNOTATIONS_DIR / "Train" / "gold_quality" / "json_format" / "train_gold.json",
    ANNOTATIONS_DIR / "Train" / "platinum_quality" / "json_format" / "train_platinum.json",
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver.json",
]

train_data = load_ner_data(train_files)
train_documents = prepare_documents_for_ner(train_data)

print(f"\nTotal training documents: {len(train_documents)}")
print(f"Total training text segments: {len(train_documents)}")

Loaded 208 documents from train_gold.json
Loaded 111 documents from train_platinum.json
Loaded 499 documents from train_silver.json

Total training documents: 1636
Total training text segments: 1636


In [5]:
# Load dev data
dev_data_path = (
    ANNOTATIONS_DIR
    / "Dev"
    / "json_format"
    / "dev.json"
)
dev_data_path = (
    ANNOTATIONS_DIR
    / "Dev"
    / "json_format"
    / "dev.json"
)

with dev_data_path.open(encoding="utf-8") as f:
    dev_data = json.load(f)

dev_documents = prepare_documents_for_ner(dev_data)

print(f"Total dev documents: {len(dev_documents)}")

Total dev documents: 80


In [6]:
# Show example document
example_doc = train_documents[10]
print(f"Example document:")
print(f"  PMID: {example_doc['pmid']}")
print(f"  Location: {example_doc['location']}")
print(f"  Text: {example_doc['text'][:200]}...")
print(f"  Number of entities: {len(example_doc['entities'])}")
print(f"\nFirst 3 entities:")
for entity in example_doc['entities'][:3]:
    print(f"    - '{entity['text_span']}' [{entity['label']}] @ {entity['start_idx']}-{entity['end_idx']}")

Example document:
  PMID: 37127945
  Location: title
  Text: A systematic review on gut-brain axis aberrations in bipolar disorder and methods of balancing the gut microbiota....
  Number of entities: 2

First 3 entities:
    - 'bipolar disorder' [DDF] @ 53-68
    - 'gut microbiota' [microbiome] @ 99-112


## Initialize BERT Model and Tokenizer

In [7]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"✓ Model loaded with {model.num_labels} labels")

# Test tokenization
sample_text = "The gut microbiome plays a role in Parkinson's disease."
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 358.72it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Tokenizer loaded: BertTokenizer
✓ Model loaded with 27 labels

Sample tokenization: ['The', 'gut', 'micro', '##bio', '##me', 'plays', 'a', 'role', 'in', 'Parkinson', "'", 's', 'disease', '.']


## BIO Tag Generation for Training Data

align_labels_with_tokens(text, entities, tokenizer, label2id)

1. Tokenizza con:
encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=True, truncation=True, max_length=512)

offset_mapping ti dà per ogni token la coppia (start_char, end_char) nel testo originale.
2. Inizializza tutti i token a O.
3. Ordina le entità per start e poi per lunghezza decrescente:
sorted_entities = sorted(entities, key=lambda e: (e['start_idx'], -(e['end_idx'] - e['start_idx'])))
✅ così in caso di overlap provi a mettere prima le più lunghe.

4. Per ogni entità cerchi quali token “overlappano” lo span:
if token_start < entity_end and token_end > entity_start:
✅ overlap robusto.

5. Applichi BIO sui token trovati MA con vincolo “non etichettare due volte” usando labeled_positions.

In [8]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    """
    Create BIO tags for tokenized text based on character-level entity annotations.

    Key fixes vs baseline:
    - Handles inclusive end_idx in dataset by converting to exclusive end for overlap checks.
    - Ignores special tokens and (optionally) can ignore subword-only labeling errors.
    - Deterministic overlap policy: longer spans first; do not overwrite already-labeled tokens.
    - Casts offsets to int for safety.
    """
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]  # list[(start,end)] end is exclusive

    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    # Initialize all labels as 'O'
    labels = ["O"] * len(input_ids)

    # Sort entities: earlier start first, then longer first (so we keep more specific spans)
    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  # ✅ dataset end_idx is inclusive → convert to exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, (tok_start, tok_end) in enumerate(offset_mapping):
            tok_start = int(tok_start)
            tok_end = int(tok_end)

            # Special tokens have (0,0) offsets in HF tokenizers
            if tok_start == 0 and tok_end == 0:
                continue

            # Robust overlap check (character spans)
            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        # Apply BIO tags if we found any overlapping tokens
        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue

                if i == ent_token_start:
                    tag = f"B-{ent_label}"
                else:
                    tag = f"I-{ent_label}"

                # Fallback safety: if tag not in label2id, keep O
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }


## Process Training and Dev Data with BIO Tags

In [9]:
# Process training data
print("Processing training data...")
processed_train = []

for i, doc in enumerate(tqdm(train_documents, desc="Processing train")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_train.append(processed)

print(f"✓ Training data processed: {len(processed_train)} segments")

Processing training data...


Processing train: 100%|██████████| 1636/1636 [00:03<00:00, 448.71it/s]

✓ Training data processed: 1636 segments


In [10]:
# Process dev data
print("Processing dev data...")
processed_dev = []

for i, doc in enumerate(tqdm(dev_documents, desc="Processing dev")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_dev.append(processed)

print(f"✓ Dev data processed: {len(processed_dev)} segments")

Processing dev data...


Processing dev: 100%|██████████| 80/80 [00:00<00:00, 440.83it/s]

✓ Dev data processed: 80 segments


In [11]:
# Show example with BIO tags
example_idx = 10
example = processed_train[example_idx]

print(f"Example from training data:")
print(f"  Text: {example['text'][:150]}...")
print(f"  Entities: {len(example['entities'])}")
print(f"\nToken-Label pairs (first 30):")

token_label_pairs = []
for token, label_id in zip(example['tokens'][:30], example['labels'][:30]):
    label = id2label[label_id]
    token_label_pairs.append((token, label))

df = pd.DataFrame(token_label_pairs, columns=['Token', 'Label'])
print(df.to_string(index=False))

Example from training data:
  Text: A systematic review on gut-brain axis aberrations in bipolar disorder and methods of balancing the gut microbiota....
  Entities: 2

Token-Label pairs (first 30):
     Token        Label
     [CLS]            O
         A            O
systematic            O
    review            O
        on            O
       gut            O
         -            O
     brain            O
      axis            O
         a            O
     ##ber            O
 ##rations            O
        in            O
        bi        B-DDF
     ##pol        I-DDF
      ##ar        I-DDF
  disorder        I-DDF
       and            O
   methods            O
        of            O
 balancing            O
       the            O
       gut B-microbiome
     micro I-microbiome
     ##bio I-microbiome
      ##ta I-microbiome
         .            O
     [SEP]            O


## Prepare Dataset for BERT Training

In [12]:
class NERDataset(Dataset):
    def __init__(self, processed_data):
        self.data = processed_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
        }

print("✓ Custom dataset class defined")

✓ Custom dataset class defined


In [13]:
# Create datasets
print("Creating training datasets...")

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

print(f"✓ Training dataset: {len(train_dataset)} examples")
print(f"✓ Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
✓ Training dataset: 1636 examples
✓ Dev dataset: 80 examples


## Configure Training Arguments

In [14]:
# Setup data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, return_tensors="pt")
print("✓ Data collator initialized")

✓ Data collator initialized


In [15]:
from collections import Counter
import numpy as np
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics_seqeval(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        seq_true = []
        seq_pred = []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id == -100:
                continue
            seq_true.append(id2label[int(l_id)])
            seq_pred.append(id2label[int(p_id)])
        true_labels.append(seq_true)
        true_preds.append(seq_pred)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    # Core optimization
    learning_rate=3e-5,                 # often better than 2e-5 for BioBERT NER
    lr_scheduler_type="linear",
    warmup_ratio=0.1,                   # critical for stability with higher LR
    weight_decay=0.01,

    # Batch/effective batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch = 16 (usually helps)

    # Training length
    num_train_epochs=5,                 # 3 is often too short for NER

    # Evaluation / checkpointing
    eval_strategy="epoch",        # use this name (more compatible than eval_strategy)
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_smoothing_factor=0.0,  # keep 0 for token classification; don't smooth rare labels away
    # If you have compute_metrics with seqeval later, use f1
    metric_for_best_model="f1", #CHANGED
    greater_is_better=True,

    # Runtime / logging
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("✓ Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Training configuration ready
  Batch size: 8
  Epochs: 5
  Learning rate: 3e-05


## Train BERT Model

Note: This cell might take several minutes to hours depending on dataset size and hardware.

**Additional configurations to test:**
- Change hyperparameters (*learning_rate*, *batch_size*, *num_train_epochs*, *weight_decay*)
- Try different pre-trained models (e.g., "allenai/scibert_scivocab_uncased", "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
- Experiment with max_length for longer contexts

In [17]:
import torch
from transformers import Trainer

def compute_class_weights(processed_train, num_labels, ignore_index=-100, power=0.5):
    """
    Compute class weights from token label counts.
    power=0.5 -> sqrt inverse frequency (usually stable).
    """
    counts = Counter()
    for ex in processed_train:
        for y in ex["labels"]:
            if y == ignore_index:
                continue
            counts[int(y)] += 1

    # build weights: w_c = (1 / freq_c)^power
    freqs = np.zeros(num_labels, dtype=np.float64)
    for c in range(num_labels):
        freqs[c] = counts.get(c, 0)

    # avoid div-by-zero for unseen classes (shouldn't happen, but safe)
    freqs[freqs == 0] = 1.0

    weights = (1.0 / freqs) ** power

    # normalize weights to mean=1 (keeps loss scale reasonable)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits  # (B, T, C)

        # flatten
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


# --- compute weights and inspect FOOD-related weights ---
class_weights = compute_class_weights(processed_train, num_labels=len(label_list), power=0.5)
class_weights = torch.clamp(class_weights, min=0.5, max=5.0) #clipping added

print("Weight(B-food) =", float(class_weights[label2id["B-food"]]))
print("Weight(I-food) =", float(class_weights[label2id["I-food"]]))
print("Weight(O)      =", float(class_weights[label2id["O"]]))
pairs = [(id2label[i], float(class_weights[i])) for i in range(len(label_list))]
pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
print("Top 10 highest weights:")
for lab, w in pairs_sorted[:10]:
    print(f"{lab:30s} {w:.3f}")



Weight(B-food) = 2.6118357181549072
Weight(I-food) = 1.7602269649505615
Weight(O)      = 0.5
Top 10 highest weights:
B-food                         2.612
B-statistical technique        1.995
B-gene                         1.919
I-food                         1.760
B-drug                         1.567
B-animal                       1.429
B-biomedical technique         1.288
B-dietary supplement           1.247
B-anatomical location          1.138
I-statistical technique        0.981


In [18]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_seqeval,
    class_weights=class_weights,
)


print("✓ Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

✓ Trainer initialized
  Training samples: 1636
  Evaluation samples: 80


In [19]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("✓ TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,3.027544,0.499201,0.546996,0.705561,0.616242
2,0.827918,0.382544,0.606680,0.811304,0.694228
3,0.623028,0.342445,0.687218,0.833181,0.753193
4,0.509144,0.334317,0.683125,0.845032,0.755501
5,0.462731,0.335965,0.687034,0.840474,0.756048


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer


✓ TRAINING COMPLETED!
Training time: 2.48 minutes


## Save Trained Model

In [20]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"✓ Model saved to: {output_model_dir}")

Saving trained model...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]

✓ Model saved to: models/bert_biomedbert_ner_label_weight


## Load Model for Inference

In [21]:
# Load the trained model for inference
print("Loading trained model for inference...")

inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.eval()

if torch.cuda.is_available():
    inference_model = inference_model.cuda()

print(f"✓ Model loaded from: {output_model_dir}")

Loading trained model for inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 483.76it/s, Materializing param=classifier.weight]                                      


✓ Model loaded from: models/bert_biomedbert_ner_label_weight


## Inference Function

predict_entities(model, tokenizer, text, id2label)
- Tokenizza con offset mapping.
- Fa argmax dei logits.
- Trasforma in label stringhe.
- Ricostruisce entità scorrendo token per token:
   -  se B-: chiude eventuale entità precedente e ne apre una nuova
   -  se I- compatibile: estende end_idx
altrimenti: chiude entità

In [22]:
import re
import numpy as np
import torch
from tqdm import tqdm
from transformers import AutoModelForTokenClassification, AutoTokenizer

# -------------------------
# 0) Load model + tokenizer
# -------------------------
print("Loading trained model for inference...")
inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)

print(f"✓ Model loaded from: {output_model_dir}")
print(f"✓ Device: {device}")

# -------------------------
# 1) Thresholds (tune later)
# -------------------------

LABEL_THRESH = {
  "DDF": 0.88,
  "bacteria": 0.88,
  "statistical technique": 0.92,
  "biomedical technique": 0.82,   # ↓ recupera recall
  "gene": 0.75,                   # ↓ recupera recall
  "food": 0.70,                   # ↓ recupera recall
  "chemical": 0.80,
  "dietary supplement": 0.85,
  "drug": 0.80,
  "microbiome": 0.78,
  "anatomical location": 0.78,
  "human": 0.70,
  "animal": 0.70
}

DEFAULT_THRESH = 0.80

# -------------------------
# 2) Simple FP filters
# -------------------------
BAD_BACTERIA = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}

def normalize_span(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def apply_simple_filters(entities):
    cleaned = []
    for e in entities:
        s = normalize_span(e["text_span"])

        # drop obvious HTML/markup garbage
        if "<" in s or ">" in s:
            continue

        # drop empty/very short spans
        if len(s) <= 1:
            continue

        # bacteria generic junk
        if e["label"] == "bacteria" and s in BAD_BACTERIA:
            continue

        cleaned.append(e)
    return cleaned

# -------------------------
# 3) Gene vs Chemical postprocess (optional but useful)
# -------------------------
GENE_LIKE = re.compile(
    r"^(il-\d+|tnf-?α|ifn-?γ|tgf-?β\d*|snca|park7|dj-1|α-?synuclein|p-?α-?synuclein)$",
    re.IGNORECASE,
)

def postprocess_gene_vs_chemical(entities):
    for e in entities:
        if e["label"] == "chemical":
            s = normalize_span(e["text_span"])
            if GENE_LIKE.match(s):
                e["label"] = "gene"
    return entities

# -------------------------
# 4) Core predictor with scores
# -------------------------
def predict_entities_with_scores(
    model,
    tokenizer,
    text: str,
    id2label: dict,
    label2id: dict,
    max_length: int = 512,
):
    """
    Returns list of entities with:
      start_idx (inclusive), end_idx (inclusive), label, text_span, score
    Score = mean token probability over the entity span.
    """
    if not text:
        return []

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=max_length,
    )

    # offsets: [1, T, 2] -> [T,2]
    offsets = enc.pop("offset_mapping")[0].cpu().numpy()

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        logits = out.logits[0]  # [T, C]
        probs = torch.softmax(logits, dim=-1)  # [T, C]
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()
        probs_cpu = probs.cpu().numpy()

    labels = [id2label[int(i)] for i in pred_ids]

    entities = []
    current = None

    for t_idx, (lab, (s, e)) in enumerate(zip(labels, offsets)):
        s = int(s); e = int(e)

        # special tokens have (0,0)
        if s == 0 and e == 0:
            continue

        # guard
        if e <= s:
            continue

        if lab.startswith("B-"):
            if current is not None:
                entities.append(current)

            ent_label = lab[2:]
            # prob for this token: use prob of B-ent_label if exists, else max
            prob_idx = label2id.get(f"B-{ent_label}", None)
            token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())

            current = {
                "start_idx": s,
                "end_idx": e - 1,  # inclusive
                "label": ent_label,
                "text_span": text[s:e],
                "_token_probs": [token_prob],
            }

        elif lab.startswith("I-") and current is not None:
            ent_label = lab[2:]
            if ent_label == current["label"]:
                prob_idx = label2id.get(f"I-{ent_label}", None)
                token_prob = float(probs_cpu[t_idx, prob_idx]) if prob_idx is not None else float(probs_cpu[t_idx].max())

                current["end_idx"] = e - 1
                current["text_span"] = text[current["start_idx"]:e]
                current["_token_probs"].append(token_prob)
            else:
                # incompatible I- : close entity
                entities.append(current)
                current = None
        else:
            if current is not None:
                entities.append(current)
                current = None

    if current is not None:
        entities.append(current)

    # add score
    for ent in entities:
        probs_list = ent.pop("_token_probs", [])
        ent["score"] = float(np.mean(probs_list)) if probs_list else 0.0

    return entities

# -------------------------
# 5) Thresholding
# -------------------------
def filter_by_threshold(entities):
    out = []
    for e in entities:
        thr = LABEL_THRESH.get(e["label"], DEFAULT_THRESH)
        if e.get("score", 0.0) >= thr:
            out.append(e)
    return out

# -------------------------
# 6) Full prediction pipeline per segment
# -------------------------
def predict_segment_entities(model, tokenizer, text, location):
    ents = predict_entities_with_scores(
        model=model,
        tokenizer=tokenizer,
        text=text,
        id2label=id2label,
        label2id=label2id,
        max_length=512,
    )

    # 1) threshold
    ents = filter_by_threshold(ents)

    # 2) simple filters
    ents = apply_simple_filters(ents)

    # 3) postprocess gene vs chemical (optional)
    ents = postprocess_gene_vs_chemical(ents)

    # attach location and remove score if you don't want it in final output
    for e in ents:
        e["location"] = location
        # If submission format doesn't accept score, uncomment:
        e.pop("score", None)

    return ents

# -------------------------
# 7) Predict on dev set
# -------------------------
print("Running inference on dev set...")

predictions = {}

for doc in tqdm(dev_documents, desc="Predicting"):
    pmid = doc["pmid"]
    location = doc["location"]
    text = doc["text"]

    predicted_entities = predict_segment_entities(inference_model, inference_tokenizer, text, location)

    if pmid not in predictions:
        predictions[pmid] = {"entities": []}
    predictions[pmid]["entities"].extend(predicted_entities)

print(f"✓ Inference completed: {len(predictions)} documents")
total_entities = sum(len(p["entities"]) for p in predictions.values())
print(f"  Total entities predicted: {total_entities}")


Loading trained model for inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 456.71it/s, Materializing param=classifier.weight]                                      


✓ Model loaded from: models/bert_biomedbert_ner_label_weight
✓ Device: cuda
Running inference on dev set...


Predicting: 100%|██████████| 80/80 [00:01<00:00, 47.13it/s]

✓ Inference completed: 40 documents
  Total entities predicted: 974


## Save Predictions

In [23]:
# Save predictions to file
output_path = "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_NER_label_weight.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"Predictions saved to {output_path}")

Predictions saved to C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_NER_label_weight.json


## Error inspection on predictions
Per-label precision/recall/F1

Confusion matrix (based on best-overlap matching)

Boundary error rate (same label, overlap, but offsets differ; also “near miss” within ±k chars)

Top false-positive strings per label

In [24]:
from collections import defaultdict
import pandas as pd
import re

# ----------------------------
# Helpers
# ----------------------------
def norm_span(s: str) -> str:
    """Normalize span text for pattern analysis (FP strings)."""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def ent_key(ent):
    # inclusive end_idx per your format
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]), str(ent["label"]))

def ent_key_no_label(ent):
    return (int(ent["start_idx"]), int(ent["end_idx"]), str(ent["location"]))

def as_span(ent):
    # return (start, end_inclusive)
    return (int(ent["start_idx"]), int(ent["end_idx"]))

def overlap_len(a_start, a_end, b_start, b_end):
    # inclusive ends
    left = max(a_start, b_start)
    right = min(a_end, b_end)
    return max(0, right - left + 1)

def iou(a_start, a_end, b_start, b_end):
    inter = overlap_len(a_start, a_end, b_start, b_end)
    if inter == 0:
        return 0.0
    a_len = a_end - a_start + 1
    b_len = b_end - b_start + 1
    union = a_len + b_len - inter
    return inter / union

def build_index(entities):
    """
    Build location-based index for quick overlap checks.
    entities: list of dicts each with start_idx/end_idx/location/label/text_span
    """
    idx = defaultdict(list)  # loc -> list[(start,end,ent)]
    for e in entities:
        s, eend = as_span(e)
        loc = str(e["location"])
        idx[loc].append((s, eend, e))
    # sort by start for mild speed-up
    for loc in idx:
        idx[loc].sort(key=lambda x: x[0])
    return idx

def best_overlap_match(gold_ent, pred_candidates, min_iou=0.1):
    """
    Return best predicted entity overlapping the gold one, by IoU (ties by overlap length).
    pred_candidates: list[(start,end,ent)]
    """
    gs, ge = as_span(gold_ent)
    best = None
    best_iou = 0.0
    best_ol = 0

    for ps, pe, pent in pred_candidates:
        ol = overlap_len(gs, ge, ps, pe)
        if ol == 0:
            continue
        score = iou(gs, ge, ps, pe)
        if score < min_iou:
            continue
        if (score > best_iou) or (score == best_iou and ol > best_ol):
            best = pent
            best_iou = score
            best_ol = ol

    return best, best_iou, best_ol


# ----------------------------
# Flatten gold + pred
# ----------------------------
def flatten_gold(dev_data):
    gold = defaultdict(list)  # pmid -> list[ent]
    for pmid, article in dev_data.items():
        for e in article["entities"]:
            gold[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return gold

def flatten_pred(predictions):
    pred = defaultdict(list)
    for pmid, obj in predictions.items():
        for e in obj.get("entities", []):
            pred[pmid].append({
                "start_idx": int(e["start_idx"]),
                "end_idx": int(e["end_idx"]),
                "location": str(e["location"]),
                "label": str(e["label"]),
                "text_span": str(e.get("text_span", "")),
            })
    return pred


gold_by_pmid = flatten_gold(dev_data)
pred_by_pmid = flatten_pred(predictions)

ALL_LABELS = sorted(set(
    [e["label"] for pmid in gold_by_pmid for e in gold_by_pmid[pmid]] +
    [e["label"] for pmid in pred_by_pmid for e in pred_by_pmid[pmid]]
))


# ----------------------------
# (1) Per-label Precision/Recall/F1
# ----------------------------
def per_label_prf(gold_by_pmid, pred_by_pmid, labels):
    gold_sets = {lab: set() for lab in labels}
    pred_sets = {lab: set() for lab in labels}

    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            k = (pmid,) + ent_key(e)  # include pmid
            gold_sets[e["label"]].add(k)

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            k = (pmid,) + ent_key(e)
            pred_sets[e["label"]].add(k)

    rows = []
    for lab in labels:
        g = gold_sets[lab]
        p = pred_sets[lab]
        tp = len(g & p)
        fp = len(p - g)
        fn = len(g - p)

        prec = tp / (tp + fp + 1e-12)
        rec = tp / (tp + fn + 1e-12)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)

        rows.append({
            "label": lab,
            "gold": len(g),
            "pred": len(p),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": prec,
            "recall": rec,
            "f1": f1,
        })

    df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
    return df

df_prf = per_label_prf(gold_by_pmid, pred_by_pmid, ALL_LABELS)
print("\n=== Per-label Precision / Recall / F1 ===")
print(df_prf.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ----------------------------
# (2) Confusion matrix (overlap-based)
# ----------------------------
def confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1):
    conf = pd.DataFrame(0, index=labels + ["<NONE>"], columns=labels + ["<NONE>"], dtype=int)

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        used_pred = set()  # track exact pred entities used in matches (by object id tuple)
        # Map gold -> best pred overlap
        for g in gold_ents:
            loc = g["location"]
            best, best_iou, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            g_lab = g["label"]

            if best is None:
                conf.loc[g_lab, "<NONE>"] += 1
            else:
                bkey = (best["start_idx"], best["end_idx"], best["location"], best["label"], best.get("text_span",""))
                used_pred.add(bkey)
                conf.loc[g_lab, best["label"]] += 1

        # Preds with no overlap-match to any gold count as <NONE> -> pred_label
        gold_idx = build_index(gold_ents)
        for p in pred_ents:
            pkey = (p["start_idx"], p["end_idx"], p["location"], p["label"], p.get("text_span",""))
            if pkey in used_pred:
                continue
            loc = p["location"]
            best_gold, _, _ = best_overlap_match(p, gold_idx.get(loc, []), min_iou=min_iou)
            if best_gold is None:
                conf.loc["<NONE>", p["label"]] += 1

    return conf

conf = confusion_matrix_overlap(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1)
print("\n=== Confusion Matrix (rows=gold, cols=pred, overlap-based) ===")
# show top-left slice if huge
print(conf.to_string())


# ----------------------------
# (3) Boundary error rate (same label, overlap but offsets differ)
#     + "near miss" within +/- k chars
# ----------------------------
def boundary_report(gold_by_pmid, pred_by_pmid, labels, min_iou=0.1, near_k=3):
    # counts per label
    exact_tp = Counter()
    boundary_mismatch = Counter()
    near_miss = Counter()

    for pmid in gold_by_pmid.keys():
        gold_ents = gold_by_pmid.get(pmid, [])
        pred_ents = pred_by_pmid.get(pmid, [])
        pred_idx = build_index(pred_ents)

        # exact label+offset TP set for fast check
        pred_exact = set((pmid,) + ent_key(e) for e in pred_ents)

        for g in gold_ents:
            lab = g["label"]
            gk = (pmid,) + ent_key(g)

            if gk in pred_exact:
                exact_tp[lab] += 1
                continue

            # find best overlapping pred
            loc = g["location"]
            best, best_iou, _ = best_overlap_match(g, pred_idx.get(loc, []), min_iou=min_iou)
            if best is None:
                continue

            # boundary mismatch: same label but not exact offsets
            if best["label"] == lab:
                boundary_mismatch[lab] += 1

                # near miss: offsets close (start/end within +/- near_k)
                if (abs(best["start_idx"] - g["start_idx"]) <= near_k) and (abs(best["end_idx"] - g["end_idx"]) <= near_k):
                    near_miss[lab] += 1

    rows = []
    for lab in labels:
        tp = exact_tp[lab]
        bm = boundary_mismatch[lab]
        nm = near_miss[lab]
        denom = tp + bm
        rate = bm / (denom + 1e-12)  # among correct-label matches, how often boundaries differ
        rows.append({
            "label": lab,
            "exact_TP": tp,
            "boundary_mismatch_same_label": bm,
            "boundary_error_rate": rate,
            f"near_miss_within_±{near_k}": nm,
        })

    df = pd.DataFrame(rows).sort_values("boundary_error_rate", ascending=False).reset_index(drop=True)
    return df

df_boundary = boundary_report(gold_by_pmid, pred_by_pmid, ALL_LABELS, min_iou=0.1, near_k=3)
print("\n=== Boundary Errors (same label overlap but offsets differ) ===")
print(df_boundary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


# ----------------------------
# (4) FP patterns: top false-positive strings per label
# ----------------------------
def fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15):
    # gold exact set per pmid for TP check
    gold_exact = set()
    for pmid, gold_ents in gold_by_pmid.items():
        for e in gold_ents:
            gold_exact.add((pmid,) + ent_key(e))

    fp_by_label = defaultdict(Counter)

    for pmid, pred_ents in pred_by_pmid.items():
        for e in pred_ents:
            pk = (pmid,) + ent_key(e)
            if pk in gold_exact:
                continue  # true positive
            # false positive
            lab = e["label"]
            fp_by_label[lab][norm_span(e.get("text_span", ""))] += 1

    # Pretty print
    for lab, counter in sorted(fp_by_label.items(), key=lambda x: sum(x[1].values()), reverse=True):
        print(f"\n=== Top FP strings for label: {lab} (total FP={sum(counter.values())}) ===")
        for span, c in counter.most_common(top_n):
            if span == "":
                span = "<EMPTY>"
            print(f"{c:>4}  {span}")

fp_patterns(gold_by_pmid, pred_by_pmid, top_n=15)



=== Per-label Precision / Recall / F1 ===
                label  gold  pred  tp  fp  fn  precision  recall     f1
                human    86    87  83   4   3     0.9540  0.9651 0.9595
           microbiome   127   129 119  10   8     0.9225  0.9370 0.9297
               animal    73    69  66   3   7     0.9565  0.9041 0.9296
                 drug    60    62  54   8   6     0.8710  0.9000 0.8852
  anatomical location    76    79  68  11   8     0.8608  0.8947 0.8774
                  DDF   379   305 289  16  90     0.9475  0.7625 0.8450
             bacteria    54    52  43   9  11     0.8269  0.7963 0.8113
                 gene    39    33  22  11  17     0.6667  0.5641 0.6111
 biomedical technique    36    23  18   5  18     0.7826  0.5000 0.6102
             chemical   131    97  69  28  62     0.7113  0.5267 0.6053
                 food    26    15  12   3  14     0.8000  0.4615 0.5854
   dietary supplement    27    22  13   9  14     0.5909  0.4815 0.5306
statistical technique

#### food label prediction inspection

In [25]:
import numpy as np
import torch

GENE_B_ID = label2id["B-food"]
FOOD_I_ID = label2id["I-food"]

def count_food_token_predictions(model, tokenizer, docs, max_length=512):
    model.eval()
    food_b = 0
    food_i = 0
    total_tokens = 0

    for doc in tqdm(docs, desc="Counting FOOD token preds"):
        enc = tokenizer(
            doc["text"],
            return_tensors="pt",
            truncation=True,
            padding=False,
            return_offsets_mapping=True,
            max_length=max_length
        )
        offsets = enc.pop("offset_mapping")[0].cpu().numpy()

        if torch.cuda.is_available():
            enc = {k: v.cuda() for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits[0]  # [seq, num_labels]
            pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()

        for pid, (s, e) in zip(pred_ids, offsets):
            # ignore special tokens
            if int(s) == 0 and int(e) == 0:
                continue
            total_tokens += 1
            if pid == GENE_B_ID:
                food_b += 1
            elif pid == FOOD_I_ID:
                food_i += 1

    return food_b, food_i, total_tokens

gene_b_pred, gene_i_pred, total_tokens = count_food_token_predictions(
    inference_model, inference_tokenizer, dev_documents
)

print("Token-level FOOD predictions on DEV:")
print("  B-food:", gene_b_pred)
print("  I-food:", gene_i_pred)
print("  Total non-special tokens:", total_tokens)
print("  % tokens predicted as FOOD:", 100 * (gene_b_pred + gene_i_pred) / max(1, total_tokens))


Counting FOOD token preds: 100%|██████████| 80/80 [00:01<00:00, 43.92it/s]

Token-level FOOD predictions on DEV:
  B-food: 14
  I-food: 55
  Total non-special tokens: 15300
  % tokens predicted as FOOD: 0.45098039215686275


## Evaluate Performance

In [26]:
# Load evaluation functions from evaluate.py concepts
def remove_duplicated_entities(predictions):
    """Remove duplicated entities from predictions."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            #key = (ent["start_idx"], ent["end_idx"], ent["location"])
            key = (ent["start_idx"], ent["end_idx"], ent["location"], ent["label"])

            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    
    if removed_count > 0:
        print(f"Removed {removed_count} duplicated entities from predictions")

def remove_overlapping_entities_eval(predictions):
    """Remove overlapping entities, keeping longest spans."""
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = groups[loc]
            group = sorted(group, key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"],
                             longest["end_idx"],
                             longest["location"]))

        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"Removed {removed_count} overlapping entities")

print("✓ Evaluation helper functions defined")

✓ Evaluation helper functions defined


In [27]:
def evaluate_ner(predictions, ground_truth):
    """Evaluate NER predictions against ground truth."""
    # Remove duplicated and overlapping entities
    remove_duplicated_entities(predictions)
    remove_overlapping_entities_eval(predictions)
    
    LEGAL_ENTITY_LABELS = [
        "anatomical location", "animal", "bacteria", "biomedical technique",
        "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
        "human", "microbiome", "statistical technique"
    ]
    
    ground_truth_NER = dict()
    count_annotated_entities_per_label = {}
    
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}
    count_true_positives_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}

    for pmid in predictions.keys():
        entities = predictions[pmid]['entities']
        
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            if label not in LEGAL_ENTITY_LABELS:
                continue

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if pmid in ground_truth_NER and entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision, recall, f1 = 0, 0, 0
    n = len(count_annotated_entities_per_label)
    for label in count_annotated_entities_per_label.keys():
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10) 
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10) 
        
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))
    
    precision = precision / n
    recall = recall / n
    f1 = f1 / n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


# Evaluate
precision, recall, f1, micro_precision, micro_recall, micro_f1 = evaluate_ner(predictions, dev_data)

print("="*60)
print("BERT NER BASELINE RESULTS")
print("="*60)
print("\nMacro-averaged Metrics:")
print(f"  Macro-Precision: {precision:.4f}")
print(f"  Macro-Recall:    {recall:.4f}")
print(f"  Macro-F1 Score:  {f1:.4f}")

print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_precision:.4f}")
print(f"  Micro-Recall:    {micro_recall:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("="*60)

BERT NER BASELINE RESULTS

Macro-averaged Metrics:
  Macro-Precision: 0.8377
  Macro-Recall:    0.6944
  Macro-F1 Score:  0.7446

Micro-averaged Metrics:
  Micro-Precision: 0.8799
  Micro-Recall:    0.7672
  Micro-F1 Score:  0.8197


## Example Predictions

In [28]:
# Show example predictions
print("Example Predictions:\n")

sample_pmids = list(dev_data.keys())[:5]

for pmid in sample_pmids:
    article = dev_data[pmid]
    pred = predictions[pmid]
    
    print(f"Document PMID: {pmid}")
    print(f"Title: {article['metadata']['title'][:100]}...")
    print(f"\nGold entities: {len(article['entities'])}")
    print(f"Predicted entities: {len(pred['entities'])}")
    
    # Show first few predicted entities
    print("\nSample predictions:")
    for entity in pred['entities'][:5]:
        print(f"  - '{entity['text_span']}' [{entity['label']}] in {entity['location']}")
    
    # Calculate match statistics
    gold_set = set()
    for entity in article['entities']:
        gold_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    pred_set = set()
    for entity in pred['entities']:
        pred_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    correct = len(gold_set & pred_set)
    missed = len(gold_set - pred_set)
    wrong = len(pred_set - gold_set)
    
    print(f"\n✓ Correct: {correct}")
    print(f"✗ Missed: {missed}")
    print(f"✗ Wrong: {wrong}")
    print("-" * 80)
    print()

Example Predictions:

Document PMID: 36532064
Title: Hypothesis of a potential BrainBiota and its relation to CNS autoimmune inflammation....

Gold entities: 19
Predicted entities: 17

Sample predictions:
  - 'CNS autoimmune inflammation' [DDF] in title
  - 'neurological diseases' [DDF] in abstract
  - 'CNS autoimmunity' [DDF] in abstract
  - 'gut microbiota' [microbiome] in abstract
  - 'patients' [human] in abstract

✓ Correct: 15
✗ Missed: 4
✗ Wrong: 2
--------------------------------------------------------------------------------

Document PMID: 37212075
Title: IgA-Biome Profiles Correlate with Clinical Parkinson's Disease Subtypes....

Gold entities: 21
Predicted entities: 17

Sample predictions:
  - 'Parkinson's disease' [DDF] in abstract
  - 'neurodegenerative disorder' [DDF] in abstract
  - 'gut microbiota' [microbiome] in abstract
  - 'secretory IgA' [chemical] in abstract
  - 'SIgA' [chemical] in abstract

✓ Correct: 11
✗ Missed: 10
✗ Wrong: 6
-------------------------------

## Analysis: Entity Distribution by Label

In [29]:
from collections import Counter

# Count entities by label in predictions
pred_label_counts = Counter()
for pmid, pred in predictions.items():
    for entity in pred['entities']:
        pred_label_counts[entity['label']] += 1

# Count entities by label in gold standard
gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article['entities']:
        gold_label_counts[entity['label']] += 1

print("Entity Distribution by Label:")
print("="*60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-"*60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-"*60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")

Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       379        305       
anatomical location       76         79        
animal                    73         69        
bacteria                  54         52        
biomedical technique      36         23        
chemical                  131        97        
dietary supplement        27         22        
drug                      60         62        
food                      26         15        
gene                      39         33        
human                     86         87        
microbiome                127        129       
statistical technique     3          1         
------------------------------------------------------------
TOTAL                     1117       974       
